# 🎯 Anotasi Strong-Supervision (SAM + Active Learning) — Telur Bebek

Tujuan: dari **weak pseudo-label** → **strong supervision** dengan cara cerdas:

1. **Pilih otomatis ~30 gambar TERSULIT** (ambiguitas terbesar) untuk dianotasi manual.
2. **Anda klik** embrio (SAM membuat mask presisi) + konfirmasi vaskular (proposal Frangi).
3. **Model menangani sisanya** — dilatih dari 30 label kuat, lalu memprediksi gambar mudah (semi-supervised).
4. **Active learning** — model mengusulkan batch berikutnya yang masih ragu.

**Kenapa cuma 30?** Telur fertil yang **samar / early-stage / sinyal konflik** adalah yang paling sulit & paling bernilai untuk dianotasi manusia. Telur infertil yang jelas kosong + telur fertil yang embrionya jelas → cukup diserahkan ke model. Ini *active learning*: anotasi di tempat yang ketidakpastiannya paling besar.

> Aktifkan **GPU**: Runtime → Change runtime type → T4 GPU.

## 0. Setup — repo, dependencies, SAM

In [ ]:
import torch, os
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
REPO='https://github.com/yumikopubangelo/duck_egg_fertility_detection.git'
PROJ='/content/duck_egg_fertility_detection'
if not os.path.exists(PROJ):
    !git clone -b pipeline {REPO} {PROJ}
else:
    # folder sudah ada: refresh KODE saja (model/mask/data tidak disentuh)
    !cd {PROJ} && git fetch origin pipeline -q && git checkout -q origin/pipeline -- scripts src/features src/clustering src/segmentation src/preprocessing configs
%cd {PROJ}


In [ ]:
# Dependencies + SAM
!pip -q install scikit-image segment-anything >/dev/null 2>&1
# Checkpoint SAM ViT-B (~375MB). Untuk lebih cepat bisa pakai MobileSAM.
SAM_CKPT='sam_vit_b_01ec64.pth'
if not os.path.exists(SAM_CKPT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/{SAM_CKPT}
print('SAM checkpoint:', os.path.exists(SAM_CKPT), os.path.getsize(SAM_CKPT)//1e6 if os.path.exists(SAM_CKPT) else 0, 'MB')


In [ ]:
# Data dari Google Drive (struktur: train/val/test/{fertile,infertile})
from google.colab import drive; drive.mount('/content/drive')
import shutil, glob
DRIVE_DATA='/content/drive/MyDrive/duck_egg_data'   # ← sesuaikan
for split in ['train','val','test']:
    s=os.path.join(DRIVE_DATA,split); d=os.path.join(PROJ,'data',split)
    if os.path.exists(s):
        if os.path.exists(d): shutil.rmtree(d)
        shutil.copytree(s,d)
print('train fertile:', len(glob.glob('data/train/fertile/*.jpg')),
      '| infertile:', len(glob.glob('data/train/infertile/*.jpg')))


## 1. Ranking ambiguitas — pilih 30 gambar tersulit

Skor ambiguitas tinggi = sinyal embrio/vaskular **dekat ambang** atau **konflik dengan label** (fertil tapi samar). Hasil divalidasi di lokal: yang terpilih adalah telur fertil *early-stage / faint* — paling susah & paling bernilai dianotasi.

In [ ]:
import cv2, numpy as np, glob, os
from skimage.filters import frangi
import warnings; warnings.filterwarnings('ignore')

def roi_inner(g, erode=18):
    _,o=cv2.threshold(cv2.GaussianBlur(g,(7,7),0),0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    r=(o>0).astype(np.uint8)
    if r.mean()>0.5: r=1-r
    r=cv2.morphologyEx(r,cv2.MORPH_OPEN,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(35,35)))
    n,l,s,_=cv2.connectedComponentsWithStats(r,8)
    if n>1: r=(l==1+int(np.argmax(s[1:,cv2.CC_STAT_AREA]))).astype(np.uint8)
    return (cv2.erode(r*255,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(erode,erode)),iterations=2)>0).astype(np.uint8)

def signals(path):
    img=cv2.imread(path)
    if img is None: return None
    img=cv2.resize(img,(384,384)); g=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY); g=cv2.createCLAHE(2.0,(8,8)).apply(g)
    roi=roi_inner(g)
    if roi.sum()<500: return None
    blur=cv2.GaussianBlur(g,(0,0),25)
    dark=((blur.astype(float)-g.astype(float))>max(6,g[roi>0].std()*0.35))&(roi>0)
    fr=frangi(g.astype(float)/255,sigmas=range(1,5),black_ridges=True)
    return dark.sum()/roi.sum(), fr[roi>0].mean()*1e3

rows=[]
for cls,lab in [('fertile',1),('infertile',0)]:
    for p in sorted(glob.glob(f'data/train/{cls}/*.jpg')):
        s=signals(p)
        if s: rows.append((p,lab,s[0],s[1]))
names=[r[0] for r in rows]; labs=np.array([r[1] for r in rows])
emb=np.array([r[2] for r in rows]); vasc=np.array([r[3] for r in rows])
nrm=lambda x:(x-x.min())/(x.max()-x.min()+1e-9)
evidence=0.6*nrm(emb)+0.4*nrm(vasc)
thr=np.median(evidence)
boundary=1-np.abs(evidence-thr)/(np.abs(evidence-thr).max()+1e-9)
disagree=nrm(np.where(labs==1,1-evidence,evidence))
ambiguity=0.5*boundary+0.5*disagree

# Anotasi manual difokuskan ke FERTIL tersulit (infertil cukup mask kosong)
fert_idx=[i for i in np.argsort(ambiguity)[::-1] if labs[i]==1][:30]
QUEUE=[names[i] for i in fert_idx]
print(f'Total {len(rows)} gambar | 30 fertil tersulit dipilih untuk anotasi manual:')
for r,i in enumerate(fert_idx,1):
    print(f'  {r:2d}. {os.path.basename(names[i]):16s} emb={emb[i]*100:4.1f}% vasc={vasc[i]:4.2f} ambig={ambiguity[i]:.3f}')


## 2. Muat SAM + helper anotasi klik

SAM (Segment Anything) membuat mask presisi hanya dari **klik titik**. Helper di bawah menampilkan gambar di kanvas; Anda klik di **embrio**, tekan SELESAI. SAM lalu menghasilkan mask embrio. Vaskular diusulkan otomatis (Frangi) dan bisa Anda terima/tolak.

In [ ]:
from segment_anything import sam_model_registry, SamPredictor
sam=sam_model_registry['vit_b'](checkpoint=SAM_CKPT).to('cuda' if torch.cuda.is_available() else 'cpu')
predictor=SamPredictor(sam)
print('SAM siap.')

# ── Click capture di Colab (kanvas JS) ──
from google.colab.output import eval_js
from IPython.display import display, HTML
import base64, json
def get_clicks(img_rgb, instruksi='Klik di EMBRIO (boleh beberapa titik), lalu SELESAI'):
    _,buf=cv2.imencode('.png',cv2.cvtColor(img_rgb,cv2.COLOR_RGB2BGR))
    b64=base64.b64encode(buf).decode()
    js=f'''
    async function ann(){{
      const img=new Image(); img.src='data:image/png;base64,{b64}';
      await new Promise(r=>img.onload=r);
      const c=document.createElement('canvas'); c.width=img.width; c.height=img.height;
      const x=c.getContext('2d'); x.drawImage(img,0,0);
      const lab=document.createElement('div'); lab.textContent='{instruksi}';
      lab.style='font:bold 16px sans-serif;color:#16a34a;margin:6px';
      document.body.appendChild(lab); document.body.appendChild(c);
      const b=document.createElement('button'); b.textContent='SELESAI';
      b.style='display:block;margin:8px;padding:8px 16px;font-size:15px'; document.body.appendChild(b);
      const pts=[];
      c.onclick=(e)=>{{const r=c.getBoundingClientRect();
        const px=Math.round((e.clientX-r.left)*c.width/r.width);
        const py=Math.round((e.clientY-r.top)*c.height/r.height);
        pts.push([px,py]); x.fillStyle='red'; x.beginPath(); x.arc(px,py,6,0,6.28); x.fill();}};
      await new Promise(r=>b.onclick=r); c.remove(); b.remove(); lab.remove();
      return JSON.stringify(pts);
    }} ann();'''
    return json.loads(eval_js(js))


## 3. Loop anotasi — klik embrio per gambar

Jalankan sel. Untuk tiap gambar: **klik di tengah embrio** (1-3 titik cukup), tekan **SELESAI**. SAM membuat mask embrio (kelas 2), Frangi mengusulkan vaskular (kelas 1). Mask 3-kelas disimpan ke `data/segmentation/train/masks/` (menimpa pseudo-label lama untuk gambar ini).

> Tip: untuk telur yang benar-benar tak ada embrio (ragu), klik **SELESAI tanpa klik** → embrio kosong, hanya vaskular.

In [ ]:
os.makedirs('data/segmentation/train/images',exist_ok=True)
os.makedirs('data/segmentation/train/masks',exist_ok=True)
ANNOT_LOG='data/segmentation/manual_annotated.txt'
done=set(open(ANNOT_LOG).read().split()) if os.path.exists(ANNOT_LOG) else set()

def frangi_vascular(g, roi):
    fr=frangi(g.astype(float)/255,sigmas=range(1,5),black_ridges=True)
    frn=np.zeros_like(fr); frn[roi>0]=fr[roi>0]
    t=np.percentile(frn[roi>0],85) if (roi>0).any() else 0
    v=((frn>t)&(roi>0)).astype(np.uint8)
    return cv2.morphologyEx(v,cv2.MORPH_OPEN,np.ones((2,2),np.uint8))

import matplotlib.pyplot as plt
for k,path in enumerate(QUEUE,1):
    stem=os.path.splitext(os.path.basename(path))[0]
    if stem in done:
        print(f'[{k}/30] {stem} sudah dianotasi, lewati'); continue
    bgr=cv2.imread(path); H0,W0=bgr.shape[:2]
    disp=cv2.resize(bgr,(512,512)); rgb=cv2.cvtColor(disp,cv2.COLOR_BGR2RGB)
    g=cv2.cvtColor(disp,cv2.COLOR_BGR2GRAY); g=cv2.createCLAHE(2.0,(8,8)).apply(g)
    roi=roi_inner(g)
    print(f'[{k}/30] {stem} — klik embrio lalu SELESAI')
    pts=get_clicks(rgb)
    # SAM embrio
    embryo=np.zeros((512,512),np.uint8)
    if pts:
        predictor.set_image(rgb)
        m,sc,_=predictor.predict(point_coords=np.array(pts),point_labels=np.ones(len(pts)),multimask_output=True)
        embryo=(m[int(np.argmax(sc))]&(roi>0)).astype(np.uint8)
    vasc=frangi_vascular(g,roi); vasc[embryo>0]=0           # embrio menimpa vaskular
    mask=np.zeros((512,512),np.uint8); mask[vasc>0]=1; mask[embryo>0]=2
    # simpan (resize ke ukuran asli, NEAREST)
    mask_full=cv2.resize(mask,(W0,H0),interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(f'data/segmentation/train/masks/{stem}.png',mask_full)
    if not os.path.exists(f'data/segmentation/train/images/{stem}.jpg'):
        shutil.copy(path,f'data/segmentation/train/images/{stem}.jpg')
    with open(ANNOT_LOG,'a') as f: f.write(stem+'\n')
    # preview
    ov=rgb.copy(); ov[vasc>0]=[230,120,30]; ov[embryo>0]=[40,200,60]
    fig,ax=plt.subplots(1,2,figsize=(7,3.6))
    ax[0].imshow(rgb); ax[0].set_title('asli'); ax[0].axis('off')
    ax[1].imshow(ov); ax[1].set_title(f'embrio={(embryo>0).mean()*100:.1f}% vasc={(vasc>0).mean()*100:.1f}%'); ax[1].axis('off')
    plt.tight_layout(); plt.show()
print('✅ Anotasi manual selesai. Mask tersimpan.')


## 4. Mask infertil = kosong (otomatis) + verifikasi

Telur infertil tidak perlu anotasi manual — mask-nya semua background (sesuai biologi: tak ada struktur).

In [ ]:
# Mask kosong untuk SEMUA infertil train (kalau belum ada)
import numpy as np
cnt=0
for p in glob.glob('data/train/infertile/*.jpg'):
    stem=os.path.splitext(os.path.basename(p))[0]
    mp=f'data/segmentation/train/masks/{stem}.png'
    img=cv2.imread(p)
    cv2.imwrite(mp, np.zeros(img.shape[:2],np.uint8))
    ip=f'data/segmentation/train/images/{stem}.jpg'
    if not os.path.exists(ip): shutil.copy(p,ip)
    cnt+=1
print(f'{cnt} mask infertil kosong dibuat.')
strong=len(open(ANNOT_LOG).read().split()) if os.path.exists(ANNOT_LOG) else 0
print(f'Strong-labeled: {strong} fertil (manual) + {cnt} infertil (kosong)')


## 5. Latih U-Net dari label kuat (weighted CE + Dice, anti-collapse)

Dilatih hanya pada gambar berlabel-kuat. Loss berbobot supaya kelas minoritas (vaskular/embrio) tidak diabaikan.

In [ ]:
import torch.nn as nn, torch.nn.functional as F, time
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import sys; sys.path.append(PROJ)
from src.segmentation.unet_lightweight import create_unet_lightweight
DEV='cuda' if torch.cuda.is_available() else 'cpu'; SZ,NC=256,3

class DS(Dataset):
    def __init__(self,aug=False):
        self.aug=aug; self.tt=T.ToTensor(); self.pairs=[]
        for ip in glob.glob('data/segmentation/train/images/*'):
            mp=f'data/segmentation/train/masks/{os.path.splitext(os.path.basename(ip))[0]}.png'
            if os.path.exists(mp): self.pairs.append((ip,mp))
    def __len__(self): return len(self.pairs)
    def __getitem__(self,i):
        ip,mp=self.pairs[i]
        im=np.array(Image.open(ip).convert('RGB').resize((SZ,SZ),Image.BILINEAR))
        mk=np.array(Image.open(mp).convert('L').resize((SZ,SZ),Image.NEAREST)).astype(np.int64)
        mk=np.clip(mk,0,NC-1)
        if self.aug and np.random.rand()<.5: im=im[:,::-1].copy(); mk=mk[:,::-1].copy()
        return self.tt(Image.fromarray(im)), torch.from_numpy(mk).long()
ds=DS(aug=True); dl=DataLoader(ds,batch_size=8,shuffle=True)
print('Training samples:',len(ds))

cnts=np.zeros(NC)
for _,y in ds:
    for k in range(NC): cnts[k]+=(y==k).sum().item()
w=1.0/(cnts/cnts.sum()+1e-6); w=np.clip(w/w.sum()*NC,0.3,8.0)
cw=torch.tensor(w,dtype=torch.float32,device=DEV); print('class weight:',w.round(2))
def dice(logits,t,eps=1.):
    p=F.softmax(logits,1); t1=F.one_hot(t,NC).permute(0,3,1,2).float()
    d=(2*(p*t1).sum((0,2,3))+eps)/(p.sum((0,2,3))+t1.sum((0,2,3))+eps); return 1-d.mean()
ce=nn.CrossEntropyLoss(weight=cw)
net=create_unet_lightweight(3,NC,True,0.2).to(DEV)
opt=torch.optim.Adam(net.parameters(),1e-3,weight_decay=1e-4)
t0=time.time()
for ep in range(1,81):
    net.train(); tl=0
    for x,y in dl:
        x,y=x.to(DEV),y.to(DEV); opt.zero_grad()
        out=net(x); loss=ce(out,y)+dice(out,y); loss.backward(); opt.step(); tl+=loss.item()*x.size(0)
    if ep%10==0 or ep==1: print(f'ep {ep:3d} loss {tl/len(ds):.3f}')
print(f'done {time.time()-t0:.0f}s')
os.makedirs('models/unet',exist_ok=True)
torch.save({'model_state_dict':net.state_dict(),'epoch':80,'metrics':{},
            'config':{'model':{'lightweight':True,'n_channels':3,'n_classes':3,'bilinear':True,'dropout_rate':0.2}}},
           'models/unet/model.pth')
print('Model tersimpan: models/unet/model.pth')


## 6. Propagasi ke gambar mudah + active-learning batch berikutnya

Model memprediksi gambar fertil yang **belum** dianotasi. Yang **percaya-diri** → pseudo-label otomatis. Yang **masih ragu** → diusulkan jadi batch anotasi manual berikutnya.

In [ ]:
net.eval(); tt=T.ToTensor()
done=set(open(ANNOT_LOG).read().split())
unl=[p for p in glob.glob('data/train/fertile/*.jpg')
     if os.path.splitext(os.path.basename(p))[0] not in done]
def predict_conf(path):
    pil=Image.open(path).convert('RGB').resize((SZ,SZ),Image.BILINEAR)
    with torch.no_grad():
        pr=torch.softmax(net(tt(pil).unsqueeze(0).to(DEV)),1)[0].cpu().numpy()
    pred=pr.argmax(0)
    # confidence = rata2 prob kelas terpilih di area non-background
    fg=pred>0
    conf=pr.max(0)[fg].mean() if fg.any() else 1.0   # tak ada fg → yakin "kosong"
    return pred, float(conf), fg.mean()
res=[]
for p in unl:
    pred,conf,fg=predict_conf(p); res.append((p,pred,conf,fg))
res.sort(key=lambda r:r[2])     # confidence terendah = paling ragu
print('=== 10 paling RAGU → usulan anotasi batch berikutnya ===')
for p,_,c,fg in res[:10]: print(f'  {os.path.basename(p):16s} conf={c:.2f} fg={fg*100:.1f}%')
# Simpan pseudo-label untuk yang percaya diri (conf>0.85)
ps=0
for p,pred,c,fg in res:
    if c>0.85:
        stem=os.path.splitext(os.path.basename(p))[0]
        full=cv2.resize(pred.astype(np.uint8),(cv2.imread(p).shape[1],cv2.imread(p).shape[0]),interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(f'data/segmentation/train/masks/{stem}.png',full)
        if not os.path.exists(f'data/segmentation/train/images/{stem}.jpg'): shutil.copy(p,f'data/segmentation/train/images/{stem}.jpg')
        ps+=1
print(f'\n{ps} pseudo-label percaya-diri disimpan. {len(res)-ps} masih ragu (anotasi lagi untuk akurasi lebih).')
NEXT_BATCH=[r[0] for r in res[:30]]
print('Set QUEUE=NEXT_BATCH lalu ulangi sel 3 untuk putaran active-learning berikutnya.')


---
## 7. Ekstraksi fitur + Train AWC (memakai U-Net hasil anotasi)

Setelah U-Net dilatih dari label kuat, kita ekstraksi fitur 322-dim (classical + deep embedding U-Net baru), lalu latih ulang **AWC**. Sel diberi pembersih NaN/Inf agar error AWC langsung terlihat.

In [ ]:
# >>> auto-sync kode dari branch pipeline (cegah awc.py/script lama) <<<
import urllib.request
_BASE='https://raw.githubusercontent.com/yumikopubangelo/duck_egg_fertility_detection/pipeline'
for _f in ['src/clustering/awc.py','scripts/05_train_awc.py','scripts/04_extract_features.py',
           'src/features/hybrid_features.py','src/features/classical_features.py','configs/awc_config.yaml']:
    urllib.request.urlretrieve(f'{_BASE}/{_f}', _f)
assert 'feature_indices' in open('src/clustering/awc.py').read(), 'awc.py gagal update'
print('kode tersinkron (awc.py OK).')

# Ekstraksi fitur memakai checkpoint U-Net hasil anotasi (models/unet/model.pth)
!python scripts/04_extract_features.py \
    --data-root data --output-dir data/features \
    --mode hybrid --unet-checkpoint models/unet/model.pth

import numpy as np
Xtr=np.load('data/features/awc_features.npy'); ytr=np.load('data/features/awc_labels.npy')
Xte=np.load('data/features/awc_test_features.npy'); yte=np.load('data/features/awc_test_labels.npy')
print('Train',Xtr.shape,'Test',Xte.shape,'| label train:',np.bincount(ytr))
print('NaN?',np.isnan(Xtr).any(),'Inf?',np.isinf(Xtr).any())


In [ ]:
# Bersihkan NaN/Inf (penyebab error AWC paling umum)
import numpy as np
for f in ['data/features/awc_features.npy','data/features/awc_test_features.npy']:
    a=np.load(f); a=np.nan_to_num(a,nan=0.0,posinf=0.0,neginf=0.0); np.save(f,a)
print('Fitur dibersihkan.')


In [ ]:
# >>> auto-sync kode dari branch pipeline (cegah awc.py/script lama) <<<
import urllib.request
_BASE='https://raw.githubusercontent.com/yumikopubangelo/duck_egg_fertility_detection/pipeline'
for _f in ['src/clustering/awc.py','scripts/05_train_awc.py','scripts/04_extract_features.py',
           'src/features/hybrid_features.py','src/features/classical_features.py','configs/awc_config.yaml']:
    urllib.request.urlretrieve(f'{_BASE}/{_f}', _f)
assert 'feature_indices' in open('src/clustering/awc.py').read(), 'awc.py gagal update'
print('kode tersinkron (awc.py OK).')

# Latih AWC (ANOVA k=20, sesuai configs/awc_config.yaml)
!python scripts/05_train_awc.py --config configs/awc_config.yaml

import json
with open('results/awc_evaluation/metrics.json') as f: m=json.load(f)
print('=== Evaluasi AWC ===')
for k,v in m['evaluation'].items():
    try: print(f'  {k}: {float(v):.4f}')
    except Exception: print(f'  {k}: {v}')


In [ ]:
# Akurasi + confusion matrix (mapping cluster->label terbaik)
import numpy as np
from itertools import permutations
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
# reload modul awc di kernel (buang cache class lama)
import importlib, src.clustering.awc as _awc; importlib.reload(_awc)
from src.clustering.awc import AdaptiveWeightedClustering

import matplotlib.pyplot as plt

model=AdaptiveWeightedClustering.load('models/awc/awc_model.pkl')
Xte=np.load('data/features/awc_test_features.npy'); yte=np.load('data/features/awc_test_labels.npy')
pred=model.predict(Xte)
best_acc,best=-1,pred
for perm in permutations(range(len(np.unique(pred)))):
    mp=np.array([perm[p] for p in pred]); a=accuracy_score(yte,mp)
    if a>best_acc: best_acc,best=a,mp
print(f'Akurasi test: {best_acc:.3f} | F1: {f1_score(yte,best,average="macro"):.3f}')
cm=confusion_matrix(yte,best)
fig,ax=plt.subplots(figsize=(4.5,4)); ax.imshow(cm,cmap='Reds')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j,i,cm[i,j],ha='center',va='center',fontsize=14,fontweight='bold',
                color='white' if cm[i,j]>cm.max()/2 else 'black')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Infertil','Fertil']); ax.set_yticklabels(['Infertil','Fertil'])
ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual'); ax.set_title(f'AWC (acc={best_acc:.1%})')
plt.tight_layout(); plt.savefig('docs/fig_awc_confusion_retrained.png',dpi=130,bbox_inches='tight'); plt.show()


In [ ]:
# Feature importance AWC (bobot adaptif)
import numpy as np, matplotlib.pyplot as plt
info=model.get_cluster_info(); fi=np.array(info['feature_importance'])
idx=np.argsort(fi)[::-1][:15]
fig,ax=plt.subplots(figsize=(9,4.5))
ax.bar(range(len(idx)),fi[idx],color='#C0392B',edgecolor='black')
ax.set_xticks(range(len(idx))); ax.set_xticklabels([f'f{i}' for i in idx],rotation=45)
ax.set_ylabel('Importance'); ax.set_title('AWC - 15 Fitur Terpenting'); ax.grid(axis='y',alpha=.3)
plt.tight_layout(); plt.savefig('docs/fig_awc_feature_importance_retrained.png',dpi=130,bbox_inches='tight'); plt.show()
print('Silhouette:',info.get('silhouette_score'),'| iterasi:',info.get('iterations'))


## 8. Simpan ke Drive

In [ ]:
OUT='/content/drive/MyDrive/duck_egg_annotation'; os.makedirs(OUT,exist_ok=True)
shutil.copy('models/unet/model.pth',OUT)
shutil.make_archive(f'{OUT}/masks_strong','zip','data/segmentation/train/masks')
shutil.copy(ANNOT_LOG,OUT)
print('Tersimpan ke',OUT,'— model + mask + log anotasi')


---
### Alur active learning (ringkas)
1. Sel 1 → pilih 30 tersulit · 2. Sel 3 → klik anotasi · 3. Sel 5 → latih · 4. Sel 6 → propagasi + usulan batch ragu
5. `QUEUE=NEXT_BATCH` → ulangi dari sel 3. Tiap putaran model makin pintar, anotasi makin sedikit.

**Untuk artikel:** "Expert-verified semi-automated annotation via SAM, prioritised by model uncertainty (active learning). N strong-labeled images; remainder pseudo-labeled with confidence > 0.85." Jujur, kuat, dan bisa dipertahankan.